# RT-DETR 物体検出・カウント

COCO 80分類で事前学習済みの RT-DETR を使い、画像内の物体を検出してクラス別に数えます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image, ImageDraw
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

MODEL_ID = 'PekingU/rtdetr_r101vd'
IMAGE_PATH = Path('../assets/test1.png')
CONFIDENCE_THRESHOLD = 0.30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'入力画像が見つかりません: {IMAGE_PATH}')

print(f'推論デバイス: {DEVICE}')
print(f'信頼度閾値: {CONFIDENCE_THRESHOLD}')

In [ ]:
image = Image.open(IMAGE_PATH).convert('RGB')
processor = RTDetrImageProcessor.from_pretrained(MODEL_ID)
model = RTDetrForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()

inputs = processor(images=image, return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    outputs = model(**inputs)

result = processor.post_process_object_detection(
    outputs,
    target_sizes=torch.tensor([image.size[::-1]], device=DEVICE),
    threshold=CONFIDENCE_THRESHOLD,
)[0]

detections = []
for score, label_id, box in zip(result['scores'], result['labels'], result['boxes']):
    x1, y1, x2, y2 = (round(value, 1) for value in box.tolist())
    detections.append({
        'class': model.config.id2label[label_id.item()],
        'confidence': round(score.item(), 3),
        'bbox': [x1, y1, x2, y2],
    })

print(f'検出数: {len(detections)}')

In [ ]:
annotated = image.copy()
draw = ImageDraw.Draw(annotated)
for detection in detections:
    x1, y1, x2, y2 = detection['bbox']
    draw.rectangle((x1, y1, x2, y2), outline='lime', width=3)
    draw.text((x1, max(0, y1 - 14)), f"{detection['class']} {detection['confidence']:.2f}", fill='lime')

plt.figure(figsize=(14, 10))
plt.imshow(annotated)
plt.axis('off')
plt.title(f'RT-DETR detections: {len(detections)}')
plt.show()

In [ ]:
details = pd.DataFrame(detections, columns=['class', 'confidence', 'bbox'])
counts = (
    details.groupby('class').size().reset_index(name='検出数').sort_values('検出数', ascending=False)
    if not details.empty else pd.DataFrame(columns=['class', '検出数'])
)

print('### クラス別の検出数')
display(counts.rename(columns={'class': 'クラス'}))
print(f'合計: {len(details)} 個')
print('### 検出詳細')
display(details)